<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 10 Geometry

The notebook judges the geometry of `bausteine-core` one primitive at a time through the devtool:
the matrices of chapter 6 of the thesis and their order in `FillTM` of `M2.bas`, the meaning of the
dimensions of each body, the polygon models of the unit bodies, the tree of the CSG with its repair
and its roles, and the examples.

In [ ]:
tolerance = 1e-4

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()

## 1. The rotation matrices of the thesis

Section 6.2.4 of the thesis and `FillTM` write the rotation about x with `a(2,3) = -sin` and
`a(3,2) = sin`, about y with `b(1,3) = sin` and `b(3,1) = -sin`, and about z with `c(1,2) = -sin` and
`c(2,1) = sin`. A cuboid of size one at the origin has exactly the rotation as its matrix.

In [ ]:
def rows(matrix16):
    # The devtool answers columns, so element (r, c) is matrix16[c * 4 + r].
    return [[matrix16[c * 4 + r] for c in range(4)] for r in range(4)]

c30, s30 = math.cos(math.radians(30)), math.sin(math.radians(30))
expected = {
    (30, 0, 0): [[1, 0, 0], [0, c30, -s30], [0, s30, c30]],
    (0, 30, 0): [[c30, 0, s30], [0, 1, 0], [-s30, 0, c30]],
    (0, 0, 30): [[c30, -s30, 0], [s30, c30, 0], [0, 0, 1]],
}
for rotation, want in expected.items():
    m = rows(devtool('matrix', packed(item('CUBOID', rotation=rotation)), 0))
    got = [row[:3] for row in m[:3]]
    check(close(got, want, tolerance), f'the rotation {rotation} is the matrix of section 6.2.4', got)

## 2. The order of the transformations

`FillTM` builds translation · rotation x · rotation y · rotation z · scaling, so a point is scaled,
turned about z, then y, then x, and moved last.

In [ ]:
def mat(r):
    return r
def mul(a, b):
    return [[sum(a[i][k] * b[k][j] for k in range(4)) for j in range(4)] for i in range(4)]
def rot(axis, deg):
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    m = [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]
    i, j = {'x': (1, 2), 'y': (2, 0), 'z': (0, 1)}[axis]
    m[i][i], m[i][j], m[j][i], m[j][j] = c, -s, s, c
    return m
def translate(x, y, z):
    return [[1, 0, 0, x], [0, 1, 0, y], [0, 0, 1, z], [0, 0, 0, 1]]
def scale(x, y, z):
    return [[x, 0, 0, 0], [0, y, 0, 0], [0, 0, z, 0], [0, 0, 0, 1]]

want = mul(mul(mul(mul(translate(1, 2, 3), rot('x', 20)), rot('y', 35)), rot('z', 50)), scale(2, 3, 4))
got = rows(devtool('matrix', packed(item('CUBOID', (1, 2, 3), (2, 3, 4), (20, 35, 50))), 0))
check(close(got, want, tolerance), 'the matrix is T · Rx · Ry · Rz · S', got)

## 3. The dimensions of the bodies

The input mask of version 1.3 asks for a radius of a sphere, a radius and a height of a cone and a
cylinder, one edge of a cube and three edges of a cuboid, a pyramid and a prism.

In [ ]:
scales = {
    'SPHERE': (2, 2, 2), 'CUBE': (2, 2, 2), 'CONE': (2, 2, 7), 'CYLINDER': (2, 2, 7),
    'DISC': (2, 2, 1), 'CUBOID': (2, 5, 7), 'PYRAMID': (2, 5, 7), 'PRISM': (2, 5, 7),
}
for kind, want in scales.items():
    m = rows(devtool('matrix', packed(item(kind, size=(2, 5, 7))), 0))
    got = (m[0][0], m[1][1], m[2][2])
    check(close(got, want), f'a {kind.lower()} of size (2, 5, 7) scales by {want}', got)

## 4. A group moves its children

The matrix of an item in a group is the matrix of the group times its own.

In [ ]:
scene = packed(item('UNION', position=(10, 0, 0), rotation=(0, 0, 90)), item('CUBE', position=(1, 0, 0), parent=0))
m = rows(devtool('matrix', scene, 1))
want = mul(mul(translate(10, 0, 0), rot('z', 90)), translate(1, 0, 0))
check(close(m, want, tolerance), 'the child of a group carries the matrix of the group', m)

## 5. The polygon models of the unit bodies

Every body but the disc is a closed surface, so no edge borders one face alone, and its Euler
characteristic V − E + F is 2. A disc is open, with one ring of border edges, or two with a hole.

In [ ]:
bounds = {
    1: ([-1, -1, -1], [1, 1, 1]), 2: ([-1, -1, 0], [1, 1, 1]), 3: ([-1, -1, 0], [1, 1, 1]),
    4: ([0, 0, 0], [1, 1, 1]), 5: ([0, 0, 0], [1, 1, 1]), 6: ([0, 0, 0], [1, 1, 1]),
    7: ([0, 0, 0], [1, 0.866025, 1]), 8: ([-1, -1, 0], [1, 1, 0]),
}
for code, (lo, hi) in bounds.items():
    m = devtool('mesh', code)
    check(close(m['min'], lo, 1e-3) and close(m['max'], hi, 1e-3), f"the unit {m['body']} spans {lo} to {hi}", m)
    if code != 8:
        euler = m['points'] - m['edges'] + m['triangles']
        check(m['borders'] == 0 and euler == 2, f"the unit {m['body']} is closed with V - E + F = 2", m)
disc = devtool('mesh', 8, 0)
ring = devtool('mesh', 8, 0.5)
check(disc['borders'] == 48 and disc['points'] - disc['edges'] + disc['triangles'] == 1, 'a disc without a hole has one border ring', disc)
check(ring['borders'] == 96 and ring['points'] - ring['edges'] + ring['triangles'] == 0, 'a disc with a hole has two border rings', ring)

## 6. The tree of the CSG

A link to an item that is no group or to the item itself is cleared, and a cycle is cut at its
first item, so a walk of the tree always ends. A later child of a difference cuts, and so does every item below it.

In [ ]:
broken = packed(
    item('CUBE', parent=0),            # its own parent
    item('UNION', parent=2),           # a cycle of 1 and 2
    item('UNION', parent=1),
    item('SPHERE', parent=0),          # a parent that is no group
)
items = devtool('decode', broken)['items']
check([i['parent'] for i in items] == [None, None, 1, None], 'every broken link is cleared, and a cycle is cut at its first item', items)

tree = packed(
    item('DIFFERENCE'),
    item('CUBOID', parent=0),
    item('UNION', parent=0),
    item('CYLINDER', parent=2),
    item('SPHERE', parent=2),
)
roles = [devtool('role', tree, i) for i in range(5)]
check([r['role'] for r in roles] == ['solid', 'solid', 'cutter', 'cutter', 'cutter'], 'the later children of a difference cut, with everything below them', roles)
check([r['depth'] for r in roles] == [0, 1, 1, 2, 2], 'the depths follow the nesting', roles)

## 7. New bodies, `#Neu` and `Neu Ende`

A declaration defines a new body from its children and places it once where it stands, as `Neu Ende`
of version 1.3 does. An instance places it again with its own transformation, and replaces the
transformation of the declaration. An instance inside the declaration it places is cut off, so the
expansion always ends.

In [ ]:
declared = packed(
    item('DECLARE'),
    item('CUBOID', size=(2, 2, 2), parent=0),
    item('INSTANCE', position=(10, 0, 0), rotation=(0, 0, 90), source=0),
)
decoded = devtool('decode', declared)['items']
check(decoded[2]['source'] == 0 and decoded[2]['code'] == 200, 'an instance names its declaration')
faces = devtool('faces', declared)
check(faces['triangles'] == 24 and faces['items'] == [1, 2], 'the cuboid draws twice, once for the declaration and once for the instance', faces)
check(close(faces['bounds'], [0, 0, 0, 10, 2, 2], 1e-3), 'the instance turns the copy by 90 degrees about z and moves it by 10', faces['bounds'])

looped = packed(item('DECLARE'), item('INSTANCE', parent=0, source=0), item('SPHERE', parent=0))
check(devtool('decode', looped)['items'][1]['source'] == 0, 'the source stays, the expansion is what ends the loop')
check(devtool('faces', looped)['items'] == [2], 'an instance inside its own declaration places nothing')

bad = packed(item('SPHERE'), item('INSTANCE', source=0))
check(devtool('decode', bad)['items'][1]['source'] is None, 'a source that is no declaration is cleared')

## 8. The examples

In [ ]:
counts = {e['key']: len(devtool('example', e['key'])['items']) for e in devtool('examples')}
check(counts == {'bausteine': 8, 'haus': 6, 'werkstueck': 3, 'kugeln': 16, 'roulette': 21}, 'the examples hold their items', counts)
werk = devtool('example', 'werkstueck')
check([i['code'] for i in werk['items']] == [102, 5, 3], 'the work piece is a cuboid minus a cylinder', werk['items'])
check(werk['items'][0]['texture'] == 'Polished_Chrome', 'the work piece wears polished chrome')